# Métriques d'évaluation en vision par ordinateur
Démo du package `cv_metrics` (NumPy, from scratch) sur la scène de rue fil rouge.

1. Classification · 2. Détection · 3. Segmentation · 4. Validation

In [ ]:
import sys, numpy as np, matplotlib.pyplot as plt
sys.path += ['.', 'examples']
from cv_metrics import *
from cv_metrics.detection import COCO_IOU_THRESHOLDS
import scene

## 1. Classification
Matrice de confusion : ligne = vraie classe, colonne = classe prédite.

In [ ]:
cm_cible = [[27, 2, 1], [3, 16, 1], [1, 2, 7]]   # urbain, autoroute, campagne
y_true = np.repeat([0, 1, 2], [30, 20, 10])
y_pred = np.concatenate([np.repeat([0, 1, 2], row) for row in cm_cible])

print(confusion_matrix(y_true, y_pred))
print('accuracy  =', round(accuracy(y_true, y_pred), 3))
print('F1 macro  =', round(precision_recall_f1(y_true, y_pred, average='macro')[2], 3))

**Piège du déséquilibre** : 950 radios saines, 50 malades.

In [ ]:
verite = np.array([0] * 950 + [1] * 50)
naif = np.zeros_like(verite)                       # répond toujours « sain »
utile = verite.copy(); utile[950:960] = 0; utile[:30] = 1
for nom, pred in [('naïf', naif), ('utile', utile)]:
    f1_malade = precision_recall_f1(verite, pred, 2, average=None)[2][1]
    print(f'{nom:6} accuracy={accuracy(verite, pred):.2f}   F1 malade={f1_malade:.2f}')

## 2. Détection
IoU = aire(intersection) / aire(union). Un TP si IoU ≥ 0,5.

In [ ]:
print(box_iou([[0, 0, 2, 2]], [[1, 0, 3, 2]]))   # 2 / 6 = 0.333

In [ ]:
%run examples/02_detection.py

In [ ]:
from IPython.display import Image
Image('outputs/detection_pr_curve.png', width=500)

## 3. Segmentation
Sémantique : on compare les cartes pixel par pixel.

In [ ]:
gt, gt_inst = scene.ground_truth()
pred, pred_inst = scene.prediction()
K = len(scene.CLASSES)
print('pixel accuracy =', round(pixel_accuracy(gt, pred), 3))
for c, iou in zip(scene.CLASSES, iou_per_class(gt, pred, K)):
    print(f'  IoU {c:9} = {iou:.3f}')
print('mIoU =', round(mean_iou(gt, pred, K), 3), '  mDice =', round(mean_dice(gt, pred, K), 3))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a, img, titre in zip(ax, [scene.colorize(gt), scene.colorize(pred), gt != pred],
                         ['Vérité terrain', 'Prédiction', 'Pixels faux']):
    a.imshow(img, cmap='Oranges'); a.set_title(titre); a.axis('off')

Instance : même mAP qu'en détection, mais avec l'IoU des **masques** (piétons 1 et 2 fusionnés).

In [ ]:
%run examples/03_segmentation.py

## 4. Validation
Chaque métrique est comparée à scikit-learn et à pycocotools (évaluateur officiel COCO).

In [ ]:
!python -m pytest -q